In [1]:
path="/data_lake/gold/intlpris/"

In [2]:
from pyspark.sql import SparkSession
from impala.dbapi import connect
import pandas as pd
import os
import json
import requests
from pyspark.sql import functions as sf, types as T
import sys
from datetime import datetime, timedelta

spark = SparkSession.builder \
            .appName("Inteligência Prisional") \
            .config("spark.dynamicAllocation.enabled", "true") \
            .config("spark.dynamicAllocation.initialExecutors", "1") \
            .config("spark.dynamicAllocation.minExecutors", "1") \
            .config("spark.dynamicAllocation.maxExecutors", "4") \
            .config("spark.executor.memory", "4g") \
            .config("spark.executor.cores", "1") \
            .config("spark.driver.memory", "4g") \
            .config("spark.driver.cores", "1") \
            .config("spark.yarn.executor.memoryOverhead", "1g") \
            .config("spark.executor.memoryOverhead", "1g") \
            .config("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY") \
            .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
            .master('local') \
            .enableHiveSupport() \
            .getOrCreate()

/opt/cloudera/jupyterhub/lib64/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [3]:
def get_dtype_df(dataframe):
    string_cols = [col for col, dtype in dataframe.dtypes if dtype == "string"]
    
    # max em BYTES (UTF-8) para cada coluna string
    if string_cols:
        df_max_bytes = (
            dataframe.select([
                sf.max(sf.length(sf.encode(sf.col(col), "UTF-8"))).alias(col)
                for col in string_cols
            ])
            .first()
            .asDict()
        )
    else:
        df_max_bytes = {}
    
    df_schema = pd.json_normalize(json.loads(dataframe.schema.json()), record_path=['fields'])[['name', 'type']]

    def adjust_dtype(name, dtype):
        if dtype == "long":
            return "bigint"
        
        if dtype == 'string':
            max_length = df_max_bytes.get(name) or 1
            
            # ajusta para CHAR(n) se for até 16 caracteres, senão usa VARCHAR(n)
            if max_length <= 16:
                return f"char({max_length})"  # garantir CHAR(1) como mínimo
            else:
                return f"varchar({max_length})" 
            
        else:
            return dtype

    df_schema['type'] = df_schema.apply(lambda x: adjust_dtype(x['name'], x['type']), axis=1)
    return df_schema

def write_impala_table_partioned(df, impala_schema, impala_table, br_path):
    dtype_df = get_dtype_df(df)
    dtypes_sting = ",\n    ".join([f"{row['name']} {row['type']}" for index, row in dtype_df.iterrows()])
    creation_query = f"""CREATE EXTERNAL TABLE {impala_schema}.{impala_table} 
                        ({dtypes_sting})
                        STORED AS PARQUET LOCATION '{br_path}'
                        """
    drop_query = f"""DROP TABLE {impala_schema}.{impala_table} """
    conn = connect(host='worker03-prod.sejus.es.gov.br', 
                   port=21050, 
                   database='bronze', 
                   auth_mechanism='GSSAPI',
                   kerberos_service_name='impala',
                   use_ssl=True,
                   ca_cert="/var/lib/cloudera-scm-agent/agent-cert/cm-auto-global_cacerts.pem")

    cursor = conn.cursor()
    print(creation_query)
    print(drop_query)
    try:
        cursor.execute(drop_query)
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
    except: 
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
        
def enviar_gold_para_postgres(nome_tabela_origem, pk_postgres):
    """
    Cria/substitui uma tabela no PostgreSQL a partir de uma tabela Spark/Hive
    e define uma chave primária no campo informado, caso pk_postgres seja informado.

    Parâmetros:
        nome_tabela_origem : str
            Nome completo da tabela no Spark, ex: "gold.sinp_pres_loc_atual"

        pk_postgres : str
            Nome do campo que será chave primária no PostgreSQL.
            Se vier "" ou None, a tabela será criada sem PK.

    Premissas:
        - a tabela de origem já existe no Spark
        - o driver JDBC do PostgreSQL está disponível no cluster
        - o schema/tabela de destino no PostgreSQL terão o mesmo nome da origem
          Ex: gold.sinp_pres_loc_atual -> schema "sinp", tabela "sinp_pres_loc_atual"
    """

    url = "jdbc:postgresql://10.242.38.126:5432/sinp_db"
    usuario = "usr_sinp"
    senha = "u9oLzKOato#nksFZ"
    driver = "org.postgresql.Driver"

    partes = nome_tabela_origem.split(".")
    if len(partes) != 2:
        raise ValueError("Informe a tabela no formato schema.tabela. Ex: gold.sinp_pres_loc_atual")

    schema_destino = "sinp"
    tabela_destino = partes[1]

    spark.catalog.clearCache()
    spark.sql(f"REFRESH TABLE {nome_tabela_origem}")
    df = spark.table(nome_tabela_origem).cache()
    df.count()

    colunas_df = df.columns

    tem_pk = pk_postgres is not None and str(pk_postgres).strip() != ""
    pk_postgres = str(pk_postgres).strip() if pk_postgres is not None else ""

    if tem_pk and pk_postgres not in colunas_df:
        raise ValueError(f"A PK '{pk_postgres}' não existe na tabela de origem. Colunas disponíveis: {colunas_df}")

    def mapear_tipo_postgres(campo):
        tipo = campo.dataType.simpleString().lower()

        if tipo.startswith("string"):
            return "varchar"
        elif tipo.startswith("int"):
            return "integer"
        elif tipo.startswith("bigint") or tipo.startswith("long"):
            return "bigint"
        elif tipo.startswith("double"):
            return "double precision"
        elif tipo.startswith("float"):
            return "real"
        elif tipo.startswith("boolean"):
            return "boolean"
        elif tipo.startswith("timestamp"):
            return "timestamp"
        elif tipo.startswith("date"):
            return "date"
        elif tipo.startswith("smallint"):
            return "smallint"
        elif tipo.startswith("decimal"):
            return tipo.replace("decimal", "numeric")
        else:
            return "text"

    ddl_colunas = []
    for campo in df.schema.fields:
        nome_coluna = campo.name
        tipo_pg = mapear_tipo_postgres(campo)
        ddl_colunas.append(f'"{nome_coluna}" {tipo_pg}')

    ddl_create_schema = f'create schema if not exists "{schema_destino}"'
    ddl_drop_table = f'drop table if exists "{schema_destino}"."{tabela_destino}"'

    if tem_pk:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)},
                constraint pk_{tabela_destino} primary key ("{pk_postgres}")
            )
        '''
    else:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)}
            )
        '''

    jvm = spark._sc._gateway.jvm
    jvm.java.lang.Class.forName(driver)

    conn = jvm.java.sql.DriverManager.getConnection(url, usuario, senha)
    stmt = conn.createStatement()

    try:
        stmt.execute(ddl_create_schema)
        stmt.execute(ddl_drop_table)
        stmt.execute(ddl_create_table)
    finally:
        stmt.close()
        conn.close()

    propriedades = {
        "user": usuario,
        "password": senha,
        "driver": driver
    }

    df.write \
        .mode("append") \
        .jdbc(
            url=url,
            table=f'{schema_destino}.{tabela_destino}',
            properties=propriedades
        )

    print(f"Tabela enviada com sucesso para o PostgreSQL: {schema_destino}.{tabela_destino}")
    if tem_pk:
        print(f"PK definida: {pk_postgres}")
    else:
        print("Tabela criada sem PK.")

In [4]:
teste = spark.sql('Select id_processo, count(distinct id_preso) as Q from bronze.infopen_presos_processos group by 1 having count(distinct id_preso)>1')
teste.show(10,False)

+-----------+---+
|id_processo|Q  |
+-----------+---+
|160235     |3  |
|176152     |2  |
|85100      |2  |
|186314     |3  |
|237810     |2  |
|35694      |20 |
|250299     |2  |
|18944      |2  |
|172616     |2  |
|65251      |2  |
+-----------+---+
only showing top 10 rows



In [5]:
teste2= spark.sql("""
SELECT *
FROM bronze.infopen_vw_presos_processos where id_processo=186314
LIMIT 100
""")
teste2.show(10,False)

+--------+-----------+------------------------------+-------------------------+------------------------+---------------+-----------------------+------------------------------------------+-----------------------------------+-----------------+---------------------+-------+-----------------------------------------------+---------+------------------------+------------+---------+--------------------------+-------------------------------+
|id_preso|id_processo|presoprocesso_situacaojuridica|presoprocesso_situacaoreu|presoprocesso_dataprisao|preso_matricula|preso_nome             |id_estabelecimentosecuritylocalizacaoatual|preso_utilizaremrelatoriosjuridicos|processo_numero  |processo_numeroantigo|id_vara|vara_nome                                      |id_artigo|artigo_nome             |id_tipocrime|id_regime|presoprocesso_qtddiaspreso|presoprocesso_qtdtotalprocessos|
+--------+-----------+------------------------------+-------------------------+------------------------+---------------+------

In [6]:
enviar_gold_para_postgres("gold.sinp_pnt_pessoa_endereco", "id_pessoa_endereco")

Tabela enviada com sucesso para o PostgreSQL: sinp.sinp_pnt_pessoa_endereco
PK definida: id_pessoa_endereco


In [7]:
from contexto import *
from pyspark.sql import functions as F
import os

# =============================================================================
# PROCESSOS - ENTIDADE PROCESSO + RELAÇÃO PRESO/PESSOA x PROCESSO
#
# Origem principal:
#   bronze.infopen_vw_presos_processos
#
# Domínios:
#   bronze.infopen_tipos_crime
#   bronze.infopen_situacoes
#   bronze.infopen_situacoes_presos
#
# Apoio pessoa:
#   gold.sinp_pnt_pessoa_preso
#
# Saídas:
#   gold.sinp_ent_processos
#   gold.sinp_rl_preso_processo
#
# Postgres:
#   sinp.sinp_ent_processos
#   sinp.sinp_rl_preso_processo
# =============================================================================

try:
    path
except NameError:
    path = "/data_lake/gold/intlpris/"

# -----------------------------------------------------------------------------
# REFRESH
# -----------------------------------------------------------------------------

tabelas_refresh = [
    "bronze.infopen_vw_presos_processos",
    "bronze.infopen_tipos_crime",
    "bronze.infopen_situacoes",
    "bronze.infopen_situacoes_presos",
    "bronze.infopen_presos",
    "gold.sinp_pnt_pessoa_preso"
]

for tabela_refresh in tabelas_refresh:
    spark.sql(f"REFRESH TABLE {tabela_refresh}")

spark.catalog.clearCache()

# =============================================================================
# 01 - ENTIDADE PROCESSO
# Apenas informações do processo, com descrição dos domínios.
# =============================================================================

tabela = "sinp_ent_processos"
path_tabela = f"{path}{tabela}"

df_sinp_ent_processos = spark.sql("""
    select distinct
        cast(v.id_processo as string) as id_processo,

        cast(v.processo_numero as string) as processo_numero,
        cast(v.processo_numeroantigo as string) as processo_numero_antigo,

        cast(v.id_vara as string) as id_vara,
        trim(regexp_replace(coalesce(v.vara_nome, ''), '\\\\s+', ' ')) as vara_nome,

        cast(v.id_artigo as string) as id_artigo,
        trim(regexp_replace(coalesce(v.artigo_nome, ''), '\\\\s+', ' ')) as artigo_nome,

        cast(v.id_tipocrime as string) as id_tipocrime,
        trim(regexp_replace(coalesce(tc.tipocrime_descricao, ''), '\\\\s+', ' ')) as tipocrime_descricao

    from bronze.infopen_vw_presos_processos v

    left join bronze.infopen_tipos_crime tc
        on tc.id_tipocrime = v.id_tipocrime

    where v.id_processo is not null
""")

spark.sql(f"DROP TABLE IF EXISTS gold.{tabela}")
os.system(f"hdfs dfs -rm -r -skipTrash {path_tabela} >/dev/null 2>&1")

df_sinp_ent_processos.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(path_tabela)

write_impala_table_partioned(
    df_sinp_ent_processos,
    "gold",
    tabela,
    path_tabela
)

spark.catalog.clearCache()
spark.sql(f"REFRESH TABLE gold.{tabela}")

print(f"[OK] gold.{tabela} criada.")

# =============================================================================
# 02 - RELAÇÃO PRESO/PESSOA x PROCESSO
#
# Ponte entre pessoa/preso e processo.
# Carrega os atributos do preso no contexto do processo.
# Inclui descrições:
#   - situação jurídica
#   - situação réu
#   - situação atual do preso
#   - tipo de crime
# =============================================================================

tabela = "sinp_rl_preso_processo"
path_tabela = f"{path}{tabela}"

df_sinp_rl_preso_processo = spark.sql("""
    select distinct
        upper(substr(md5(concat_ws('|',
            coalesce(cast(pp.id_pessoa as string), '[NULL]'),
            coalesce(cast(v.id_preso as string), '[NULL]'),
            coalesce(cast(v.id_processo as string), '[NULL]')
        )), 1, 30)) as id_pessoa_processo,

        cast(pp.id_pessoa as string) as id_pessoa,
        cast(v.id_preso as string) as id_preso,
        cast(v.id_processo as string) as id_processo,

        cast(v.preso_matricula as string) as preso_matricula,
        trim(regexp_replace(coalesce(v.preso_nome, ''), '\\\\s+', ' ')) as preso_nome,

        cast(p.id_situacaopreso as string) as id_situacaopreso,
        trim(regexp_replace(coalesce(sp.situacaopreso_descricao, ''), '\\\\s+', ' ')) as situacaopreso_descricao,

        cast(v.presoprocesso_situacaojuridica as string) as id_situacao_juridica,
        trim(regexp_replace(coalesce(sj.situacao_descricao, ''), '\\\\s+', ' ')) as situacao_juridica_descricao,

        cast(v.presoprocesso_situacaoreu as string) as id_situacao_reu,
        trim(regexp_replace(coalesce(sr.situacao_descricao, ''), '\\\\s+', ' ')) as situacao_reu_descricao,

        to_timestamp(v.presoprocesso_dataprisao) as presoprocesso_dataprisao,

        cast(v.id_estabelecimentosecuritylocalizacaoatual as string) as id_estabelecimento_atual,
        cast(v.preso_utilizaremrelatoriosjuridicos as string) as preso_utilizaremrelatoriosjuridicos,

        cast(v.id_artigo as string) as id_artigo,
        trim(regexp_replace(coalesce(v.artigo_nome, ''), '\\\\s+', ' ')) as artigo_nome,

        cast(v.id_tipocrime as string) as id_tipocrime,
        trim(regexp_replace(coalesce(tc.tipocrime_descricao, ''), '\\\\s+', ' ')) as tipocrime_descricao,

        cast(v.id_regime as string) as id_regime,
        cast(v.presoprocesso_qtddiaspreso as int) as presoprocesso_qtddiaspreso,
        cast(v.presoprocesso_qtdtotalprocessos as int) as presoprocesso_qtdtotalprocessos,

        case
            when pp.id_pessoa is not null then 'S'
            else 'N'
        end as fl_id_pessoa_encontrado

    from bronze.infopen_vw_presos_processos v

    left join gold.sinp_pnt_pessoa_preso pp
        on cast(pp.id_preso as string) = cast(v.id_preso as string)

    left join bronze.infopen_presos p
        on p.id_preso = v.id_preso

    left join bronze.infopen_situacoes_presos sp
        on sp.id_situacaopreso = p.id_situacaopreso

    left join bronze.infopen_situacoes sj
        on sj.id_situacao = v.presoprocesso_situacaojuridica

    left join bronze.infopen_situacoes sr
        on sr.id_situacao = v.presoprocesso_situacaoreu

    left join bronze.infopen_tipos_crime tc
        on tc.id_tipocrime = v.id_tipocrime

    where v.id_processo is not null
      and v.id_preso is not null
""")

spark.sql(f"DROP TABLE IF EXISTS gold.{tabela}")
os.system(f"hdfs dfs -rm -r -skipTrash {path_tabela} >/dev/null 2>&1")

df_sinp_rl_preso_processo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 2000) \
    .option("compression", "snappy") \
    .parquet(path_tabela)

write_impala_table_partioned(
    df_sinp_rl_preso_processo,
    "gold",
    tabela,
    path_tabela
)

spark.catalog.clearCache()
spark.sql(f"REFRESH TABLE gold.{tabela}")

print(f"[OK] gold.{tabela} criada.")

# =============================================================================
# 03 - ENVIO AO POSTGRES
# =============================================================================

print("[POSTGRES][INICIO] gold.sinp_ent_processos", flush=True)
enviar_gold_para_postgres("gold.sinp_ent_processos", "id_processo")
print("[POSTGRES][FIM] gold.sinp_ent_processos", flush=True)

print("[POSTGRES][INICIO] gold.sinp_rl_preso_processo", flush=True)
enviar_gold_para_postgres("gold.sinp_rl_preso_processo", "id_pessoa_processo")
print("[POSTGRES][FIM] gold.sinp_rl_preso_processo", flush=True)

print("[OK] Processo finalizado.")

[OK] gold.sinp_ent_processos criada.
[OK] gold.sinp_rl_preso_processo criada.
[POSTGRES][INICIO] gold.sinp_ent_processos
[POSTGRES] sinp.sinp_ent_processos | linhas=265868
[POSTGRES][FIM] gold.sinp_ent_processos
[POSTGRES][INICIO] gold.sinp_rl_preso_processo
[POSTGRES] sinp.sinp_rl_preso_processo | linhas=319800
[POSTGRES][FIM] gold.sinp_rl_preso_processo
[OK] Processo finalizado.
